In [1]:
from kaggle_secrets import UserSecretsClient
secret_label = "Token"
token = UserSecretsClient().get_secret(secret_label)

! git clone https://{token}@github.com/PietroGhersetich/RecSys_challenge_2025

Cloning into 'RecSys_challenge_2025'...
remote: Enumerating objects: 1391, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 1391 (delta 15), reused 0 (delta 0), pack-reused 1359 (from 1)
Receiving objects: 100% (1391/1391), 625.38 MiB | 19.48 MiB/s, done.
Resolving deltas: 100% (843/843), done.
Updating files: 100% (219/219), done.


In [2]:
! pip install PyGithub requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.7/432.7 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 39.3 MB/s eta 0:00:00


In [3]:
! cd /kaggle/working/RecSys_challenge_2025 && python run_compile_all_cython.py

run_compile_all_cython: Found 11 Cython files in 5 folders...
run_compile_all_cython: All files will be compiled using your current python environment: '/usr/bin/python3'
Compiling [1/11]: MatrixFactorizationImpressions_Cython_Epoch.pyx... 
In file included from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/ndarraytypes.h:1929,
                 from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/ndarrayobject.h:12,
                 from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/arrayobject.h:5,
                 from MatrixFactorizationImpressions_Cython_Epoch.c:1252:
/usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/npy_1_7_deprecated_api.h:17:2: warning: #warning "Using deprecated NumPy API, disable it with " "#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION" []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wcpp-Wcpp]8;;]
   17 | #warning "Using deprecated NumPy API, disable it with " \
 

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import random
import tqdm
import gc
import time
import os
import shutil
import optuna
from optuna.distributions import FloatDistribution, IntDistribution, CategoricalDistribution
import scipy.sparse as sps

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/recommender-systems-2025-challenge-polimi/data_train.csv
/kaggle/input/recommender-systems-2025-challenge-polimi/sample_submission.csv
/kaggle/input/recommender-systems-2025-challenge-polimi/data_target_users_test.csv


In [5]:
from github import Github
from github import Auth

auth = Auth.Token(token)
g = Github(auth=auth)

repo = None
for r in g.get_user().get_repos():
    if r.name == 'RecSys_challenge_2025':
        repo = r
        print('Repository found')

Repository found


In [6]:
def upload_file(filepath_kaggle, filepath_github, commit_message):
    try:
        contents = repo.get_contents(filepath_github)
        with open(filepath_kaggle, "rb") as file:
            repo.update_file(contents.path, commit_message, file.read(), contents.sha)
    except Exception:
        with open(filepath_kaggle, "rb") as file:
            repo.create_file(filepath_github, commit_message, file.read())

In [7]:
db_name = 'ials_study.db'
json_name = 'ials_best_params.json'

config = {
    'tune_parameters': True,
    'database_name': db_name,
    'database_path': f'/kaggle/working/{db_name}',
    'save_json': True,
    'json_name': json_name,
    'copy_prev_best_params': False,
    'save_github': True,
    'candidates': '20',
    'n_trials': 15
}

In [8]:
try:
    shutil.copyfile(
        f'/kaggle/working/RecSys_challenge_2025/tuning_results/recall@{config["candidates"]}/{config["database_name"]}', 
        config['database_path']
    )
except FileNotFoundError:
    pass # if not present optuna will create it

# Data Import

In [9]:
train_data = pd.read_csv('/kaggle/input/recommender-systems-2025-challenge-polimi/data_train.csv')
user_ids_test = pd.read_csv('/kaggle/input/recommender-systems-2025-challenge-polimi/data_target_users_test.csv')

In [10]:
data = np.ones(len(train_data))

URM_all = sps.csr_matrix((data, (train_data['row'].values, train_data['col'].values)),
                         shape=(train_data['row'].max() + 1, train_data['col'].max() + 1))

URM_all

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3043058 stored elements and shape (27095, 6969)>

In [11]:
%cd /kaggle/working/RecSys_challenge_2025

/kaggle/working/RecSys_challenge_2025


In [12]:
from Recommenders.BaseRecommender import BaseRecommender
from Recommenders.MatrixFactorization.IALSRecommender import IALSRecommender
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample

from Evaluation.Evaluator import EvaluatorHoldout

np.int = int
np.random.seed(42)
np.bool = bool
np.float = float

In [13]:
URM_train, URM_validation = split_train_in_two_percentage_global_sample(URM_all, train_percentage = 0.8)

In [14]:
import sqlite3

def print_all_rows(table_name, database_path):
    connection = sqlite3.connect(database_path)
    cursor = connection.cursor()

    try:
        cursor.execute(f"SELECT * FROM {table_name};")
        rows = cursor.fetchall()

        print(f"All rows from table '{table_name}':")
        for row in rows:
            print(row)

    except sqlite3.Error as e:
        print(f"Error reading data from table '{table_name}': {e}")

    finally:
        cursor.close()
        connection.close()

print_all_rows('studies', config['database_path'])

All rows from table 'studies':
(1, 'item_ials_Optimization')


In [15]:
evaluator = EvaluatorHoldout(URM_validation, cutoff_list=[int(config['candidates'])])

def objective_function_ials(trial):
    params = {
        'num_factors': trial.suggest_int('num_factors', 5, 300), 
        'reg': trial.suggest_float('reg', 1e-5, 100, log=True),
        'alpha': trial.suggest_float('alpha', 1e-3, 50.0, log=True),
        'confidence_scaling': trial.suggest_categorical('confidence_scaling', ['linear', 'log'])
    }
    
    if params['confidence_scaling'] == 'log':
        params['epsilon'] = trial.suggest_float('epsilon', 1e-3, 10.0, log=True)

    evaluator_validation = EvaluatorHoldout(URM_validation, cutoff_list=[int(config['candidates'])])
    early_stopping_params = {
        'epochs_min': 5,
        'stop_on_validation': True,
        'validation_every_n': 5,
        'validation_metric': 'RECALL',
        'lower_validations_allowed': 5,
        'evaluator_object': evaluator_validation
    }

    recommender = IALSRecommender(URM_train)
    recommender.fit(**params, **early_stopping_params)
    
    epochs = recommender.get_early_stopping_final_epochs_dict()["epochs"]
    trial.set_user_attr("epochs", epochs) 

    result_df, _ = evaluator.evaluateRecommender(recommender)
    
    return result_df.loc[int(config['candidates']), 'RECALL']

EvaluatorHoldout: Ignoring 28 ( 0.1%) Users that have less than 1 test interactions


In [16]:
%%time

if config['tune_parameters']:
    study = optuna.create_study(direction='maximize', study_name='item_ials_Optimization', 
                                storage=f'sqlite:///{config["database_path"]}', load_if_exists=True)
    print(f"Previous trials found: {len(study.trials)}")
    study.optimize(objective_function_ials, n_trials=config['n_trials'])

[I 2025-12-28 20:01:59,922] Using an existing study with name 'item_ials_Optimization' instead of creating a new one.


Previous trials found: 101
EvaluatorHoldout: Ignoring 28 ( 0.1%) Users that have less than 1 test interactions
IALSRecommender: Epoch 1 of 300. Elapsed time 9.44 sec
IALSRecommender: Epoch 2 of 300. Elapsed time 18.80 sec
IALSRecommender: Epoch 3 of 300. Elapsed time 28.03 sec
IALSRecommender: Epoch 4 of 300. Elapsed time 37.25 sec
IALSRecommender: Validation begins...
EvaluatorHoldout: Processed 27067 (100.0%) in 15.53 sec. Users per second: 1743
IALSRecommender: CUTOFF: 20 - PRECISION: 0.1710847, PRECISION_RECALL_MIN_DEN: 0.2708946, RECALL: 0.2309817, MAP: 0.0688138, MAP_MIN_DEN: 0.1059253, MRR: 0.4421743, NDCG: 0.2459195, F1: 0.1965717, HIT_RATE: 0.9371559, ARHR_ALL_HITS: 0.7477330, NOVELTY: 0.0283572, AVERAGE_POPULARITY: 0.3105983, DIVERSITY_MEAN_INTER_LIST: 0.9427582, DIVERSITY_HERFINDAHL: 0.9971362, COVERAGE_ITEM: 0.3333333, COVERAGE_ITEM_HIT: 0.2733534, ITEMS_IN_GT: 0.9991390, COVERAGE_USER: 0.9989666, COVERAGE_USER_HIT: 0.9361875, USERS_IN_GT: 0.9989666, DIVERSITY_GINI: 0.06980

[I 2025-12-28 20:11:24,886] Trial 101 finished with value: 0.2383009041988969 and parameters: {'num_factors': 73, 'reg': 8.975903498015292e-05, 'alpha': 14.665506041850056, 'confidence_scaling': 'linear'}. Best is trial 42 with value: 0.24618389439636623.


EvaluatorHoldout: Ignoring 28 ( 0.1%) Users that have less than 1 test interactions
IALSRecommender: Epoch 1 of 300. Elapsed time 14.71 sec
IALSRecommender: Epoch 2 of 300. Elapsed time 29.53 sec
IALSRecommender: Epoch 3 of 300. Elapsed time 44.48 sec
IALSRecommender: Epoch 4 of 300. Elapsed time 59.54 sec
IALSRecommender: Validation begins...
EvaluatorHoldout: Processed 27067 (100.0%) in 15.69 sec. Users per second: 1725
IALSRecommender: CUTOFF: 20 - PRECISION: 0.1769184, PRECISION_RECALL_MIN_DEN: 0.2770118, RECALL: 0.2345483, MAP: 0.0741455, MAP_MIN_DEN: 0.1131637, MRR: 0.4642817, NDCG: 0.2564798, F1: 0.2016975, HIT_RATE: 0.9382643, ARHR_ALL_HITS: 0.7937044, NOVELTY: 0.0286621, AVERAGE_POPULARITY: 0.2849061, DIVERSITY_MEAN_INTER_LIST: 0.9543968, DIVERSITY_HERFINDAHL: 0.9977181, COVERAGE_ITEM: 0.3056393, COVERAGE_ITEM_HIT: 0.2565648, ITEMS_IN_GT: 0.9991390, COVERAGE_USER: 0.9989666, COVERAGE_USER_HIT: 0.9372947, USERS_IN_GT: 0.9989666, DIVERSITY_GINI: 0.0732666, SHANNON_ENTROPY: 9.406

[I 2025-12-28 20:50:41,050] Trial 102 finished with value: 0.24578703529532983 and parameters: {'num_factors': 103, 'reg': 3.674173168854987e-05, 'alpha': 7.588112887081359, 'confidence_scaling': 'linear'}. Best is trial 42 with value: 0.24618389439636623.


...

EvaluatorHoldout: Ignoring 28 ( 0.1%) Users that have less than 1 test interactions
IALSRecommender: Epoch 1 of 300. Elapsed time 12.91 sec
IALSRecommender: Epoch 2 of 300. Elapsed time 26.44 sec
IALSRecommender: Epoch 3 of 300. Elapsed time 39.64 sec
IALSRecommender: Epoch 4 of 300. Elapsed time 52.62 sec
IALSRecommender: Validation begins...
EvaluatorHoldout: Processed 27067 (100.0%) in 15.96 sec. Users per second: 1696
IALSRecommender: CUTOFF: 20 - PRECISION: 0.1682288, PRECISION_RECALL_MIN_DEN: 0.2683779, RECALL: 0.2297754, MAP: 0.0668579, MAP_MIN_DEN: 0.1041671, MRR: 0.4379491, NDCG: 0.2429213, F1: 0.1942434, HIT_RATE: 0.9347176, ARHR_ALL_HITS: 0.7344884, NOVELTY: 0.0285938, AVERAGE_POPULARITY: 0.2962241, DIVERSITY_MEAN_INTER_LIST: 0.9495464, DIVERSITY_HERFINDAHL: 0.9974756, COVERAGE_ITEM: 0.3534223, COVERAGE_ITEM_HIT: 0.2891376, ITEMS_IN_GT: 0.9991390, COVERAGE_USER: 0.9989666, COVERAGE_USER_HIT: 0.9337516, USERS_IN_GT: 0.9989666, DIVERSITY_GINI: 0.0764946, SHANNON_ENTROPY: 9.424

[I 2025-12-29 01:01:39,400] Trial 114 finished with value: 0.2409919740667177 and parameters: {'num_factors': 94, 'reg': 1.2126307226151958e-05, 'alpha': 16.105901158429987, 'confidence_scaling': 'linear'}. Best is trial 42 with value: 0.24618389439636623.


EvaluatorHoldout: Ignoring 28 ( 0.1%) Users that have less than 1 test interactions
IALSRecommender: Epoch 1 of 300. Elapsed time 13.43 sec
IALSRecommender: Epoch 2 of 300. Elapsed time 26.83 sec
IALSRecommender: Epoch 3 of 300. Elapsed time 40.24 sec
IALSRecommender: Epoch 4 of 300. Elapsed time 53.68 sec
IALSRecommender: Validation begins...
EvaluatorHoldout: Processed 27067 (100.0%) in 15.90 sec. Users per second: 1702
IALSRecommender: CUTOFF: 20 - PRECISION: 0.1738408, PRECISION_RECALL_MIN_DEN: 0.2739483, RECALL: 0.2329715, MAP: 0.0713132, MAP_MIN_DEN: 0.1098443, MRR: 0.4525977, NDCG: 0.2513593, F1: 0.1991088, HIT_RATE: 0.9373776, ARHR_ALL_HITS: 0.7694641, NOVELTY: 0.0286221, AVERAGE_POPULARITY: 0.2904654, DIVERSITY_MEAN_INTER_LIST: 0.9522881, DIVERSITY_HERFINDAHL: 0.9976126, COVERAGE_ITEM: 0.3245803, COVERAGE_ITEM_HIT: 0.2712010, ITEMS_IN_GT: 0.9991390, COVERAGE_USER: 0.9989666, COVERAGE_USER_HIT: 0.9364089, USERS_IN_GT: 0.9989666, DIVERSITY_GINI: 0.0743984, SHANNON_ENTROPY: 9.411

[I 2025-12-29 01:37:29,129] Trial 115 finished with value: 0.24573337100079887 and parameters: {'num_factors': 100, 'reg': 0.0009436400289654341, 'alpha': 10.54868117612332, 'confidence_scaling': 'linear'}. Best is trial 42 with value: 0.24618389439636623.


CPU times: user 9h 57min 34s, sys: 7min 44s, total: 10h 5min 19s
Wall time: 5h 35min 31s


In [17]:
if config['save_github'] and config['tune_parameters']:
    upload_file(
        config['database_path'], 
        f'tuning_results/recall@{config["candidates"]}/{config["database_name"]}',
        'Tuning db updated results (from kaggle notebook)'
    )

    if config['save_json']:
        j_name = config['json_name']
        l_path = f"/kaggle/working/{j_name}"
        r_path = f"tuning_results/recall@{config['candidates']}/best_params/{j_name}"
        
        best_val, best_params, best_trial_num, best_study_name, best_epochs = -float('inf'), {}, None, None, None
        
        try:
            for s in optuna.get_all_study_summaries(storage=f"sqlite:///{config['database_path']}"):
                if s.best_trial and s.best_trial.value > best_val:
                    best_val, best_params, best_trial_num, best_study_name, best_epochs = s.best_trial.value, s.best_trial.params, s.best_trial.number, s.study_name, s.best_trial.user_attrs.get("epochs")
        except:
            best_val, best_params, best_trial_num, best_study_name, best_epochs = study.best_value, study.best_params, study.best_trial.number, study.study_name, study.best_trial.user_attrs.get("epochs")

        if best_epochs is not None:
            best_params['epochs'] = int(best_epochs)
        
        prev_val, exists = -float('inf'), False
        try:
            prev_val = json.loads(repo.get_contents(r_path).decoded_content.decode()).get('value', -float('inf'))
            exists = True
        except:
            pass
    
        if not exists or best_val > prev_val:
            print(f"Updating JSON. Best: {best_val:.6f} (Prev: {prev_val:.6f})")
            with open(l_path, 'w') as f:
                json.dump({"value": best_val, "study_name": best_study_name, "trial_number": best_trial_num, "params": best_params}, f, indent=4)
            if config.get('save_github'):
                upload_file(l_path, r_path, f'Update params {j_name} (from kaggle notebook)')
        else:
            print(f"No update needed. Best: {best_val:.6f} <= Saved: {prev_val:.6f}")

No update needed. Best: 0.246184 <= Saved: 0.246184
